# VIS4AI · Notebook 01 — Setup and first run

Goal of this notebook: get everything working and **let both models solve the same GSM8K math problems**.
No metrics or plots yet — just a clean setup and the raw answers, saved to Google Drive.

| Model | Role |
|---|---|
| `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B` | main model: a small *reasoning* model that "thinks" in `<think> … </think>` before answering |
| `Qwen/Qwen2.5-Math-1.5B` | baseline: the math base model that R1-Distill was fine-tuned from (no long reasoning) |

### How to run
1. `Runtime → Change runtime type → T4 GPU`
2. `Runtime → Run all` (allow Google Drive access when asked)
3. **Disconnected?** Set `RESUME_RUN` in the settings to the name of the run folder and *Run all* again. Answers are saved after every batch, and problems that are already done are skipped.

## 1. Setup
Install packages, check the GPU and connect Google Drive (where all results are stored).

*You may see a warning "You are sending unauthenticated requests to the HF Hub". It is harmless: the models and data are public.*

In [1]:
!pip install -q "transformers>=4.56" "accelerate>=0.33" "datasets>=2.20"

In [2]:
import gc
import json
import math
import os
import re
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
import torch
import transformers
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, StoppingCriteria, StoppingCriteriaList, set_seed

assert torch.cuda.is_available(), "No GPU! Go to Runtime -> Change runtime type -> T4 GPU."
print("GPU:", torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount("/content/drive")

GPU: Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Settings
All settings are in this one cell. Each model is run the way its authors recommend, so the settings differ per model:

| | R1-Distill | Qwen2.5-Math |
|---|---|---|
| prompt | the question + instruction as one user message in the model's chat template, no system prompt (DeepSeek) | the **same text, but as plain text without a chat template** (`chat_template=False`). It is a base model and never learned the chat format: with it, it wrote junk tokens where the chat turn should end and often looped. Base Qwen2.5-Math is known to work best with no template (Liu et al., 2025, *Understanding R1-Zero-Like Training*) |
| decoding | sampling, temperature 0.6, top-p 0.95 (DeepSeek; greedy can make it loop) | greedy (Qwen) |
| `max_new_tokens` | 8192 — its thinking is long; with 2048, 1 in 5 answers was cut off | 2048 — answers are short but sometimes longer than 1024 |
| `stop_after_answer` | no — the model ends by itself after `</think>` + answer | **yes** — as a base model it does not stop by itself and starts inventing new questions after its answer, so we stop it as soon as it has written a `\boxed{...}` answer |
| answer = | the last `\boxed{...}` **after** `</think>` (`thinks=True`). No `</think>` = still thinking when the token limit hit = no answer | the first non-empty `\boxed{...}` (later boxes would belong to invented questions) |

* `N_PROBLEMS`: how many GSM8K problems to try. Raising it later only runs the new problems.
* `RESUME_RUN`: `None` starts a fresh run in a new folder `run_<i>_<date>_<time>`, so old and new answers are never mixed. **To resume after a disconnect, set it to that folder's name** (e.g. `"run_3_2026-11-08_14-30"`).
* All settings are written to `parameters.txt` in the run folder.

In [3]:
MODELS = {
    "r1-distill": dict(name="deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B", thinks=True,
                       chat_template=True,  sampling=True,  max_new_tokens=8192, stop_after_answer=False),
    "qwen-math":  dict(name="Qwen/Qwen2.5-Math-1.5B", thinks=False,
                       chat_template=False, sampling=False, max_new_tokens=2048, stop_after_answer=True),
}

N_PROBLEMS  = 50       # number of GSM8K test problems
BATCH_SIZE  = 12       # problems generated at the same time (12 fits on a T4)
TEMPERATURE = 0.6      # only used by models with sampling=True
TOP_P       = 0.95
SEED        = 42

INSTRUCTION = "Please reason step by step, and put your final answer within \\boxed{}."

RESULTS_ROOT = Path("/content/drive/MyDrive/VIS4AI/results")
RESUME_RUN   = None    # None = new run; set to a run folder name (e.g. "run_3_2026-11-08_14-30") to resume it

In [4]:
# Create a new run folder run_<i>_<date>_<time> (or reopen RESUME_RUN) and save the settings in it
if RESUME_RUN:
    RESULTS_DIR = RESULTS_ROOT / RESUME_RUN
    assert RESULTS_DIR.exists(), f"{RESULTS_DIR} does not exist, cannot resume it"
else:
    run_ids = [int(m.group(1)) for d in RESULTS_ROOT.glob("run_*") if (m := re.match(r"run_(\d+)_", d.name))]
    RESULTS_DIR = RESULTS_ROOT / f"run_{max(run_ids, default=0) + 1}_{datetime.now():%Y-%m-%d_%H-%M}"
    RESULTS_DIR.mkdir(parents=True)

parameters = dict(run=RESULTS_DIR.name, written=f"{datetime.now():%Y-%m-%d %H:%M:%S}", models=MODELS,
                  n_problems=N_PROBLEMS, batch_size=BATCH_SIZE, temperature=TEMPERATURE, top_p=TOP_P, seed=SEED,
                  instruction=INSTRUCTION, gpu=torch.cuda.get_device_name(0),
                  torch=torch.__version__, transformers=transformers.__version__)
(RESULTS_DIR / "parameters.txt").write_text("".join(f"{k}: {v}\n" for k, v in parameters.items()), encoding="utf-8")
print("Results are saved in:", RESULTS_DIR)

Results are saved in: /content/drive/MyDrive/VIS4AI/results/run_4_2026-11-08_15-40


## 2. Load the dataset
[GSM8K](https://huggingface.co/datasets/openai/gsm8k) test split: 1,319 grade-school math word problems. The correct answer is the number after `####` in the solution text.

In [5]:
gsm8k = load_dataset("openai/gsm8k", "main", split="test")

problems = [
    {"id": i,
     "question": gsm8k[i]["question"],
     "gold": gsm8k[i]["answer"].split("####")[-1].strip().replace(",", "")}
    for i in range(N_PROBLEMS)
]

print(f"{len(problems)} problems loaded. Example:\n")
print(problems[0]["question"])
print("\nCorrect answer:", problems[0]["gold"])

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


50 problems loaded. Example:

Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

Correct answer: 18


## 3. Helper functions
* **`build_prompt`** — the question plus *"Please reason step by step, and put your final answer within \boxed{}."* With `chat_template=True` the tokenizer's chat template wraps it in the model-specific special tokens; otherwise it is plain text.
* **`extract_answer`** — gets the answer out of the output, following the "answer =" row of the settings table. Empty `\boxed{}` (a repeated instruction) is ignored. `None` means no answer.
* **`load_results` / `save_results`** — read/append a `.jsonl` file on Drive (one line per problem), which makes the run resumable.
* **`load_model` / `free_gpu`** — load a model in float16 (the T4 has no bfloat16) and free the GPU afterwards.
* **`generate_answers`** — let the model answer a batch of questions. Also records how many tokens it wrote and whether it **finished** (`False` = it hit the token limit and was cut off), and the **time** until that answer was done (the batch keeps running until its longest answer ends).
* **`run_model`** — runs one model on all problems that are not done yet, saving after every batch.

In [6]:
def build_prompt(tokenizer, question, chat_template, instruction):
    """Question + instruction, wrapped in the chat template if the model uses one."""
    text = question + "\n" + instruction
    if not chat_template:
        return text + "\n"
    messages = [{"role": "user", "content": text}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def boxed_answers(text):
    """Contents of all non-empty \\boxed{...} in the text, in order (handles nested braces like \\boxed{\\frac{1}{2}})."""
    answers = []
    for match in re.finditer(r"\\boxed\{", text):
        depth = 1
        for j in range(match.end(), len(text)):
            depth += {"{": 1, "}": -1}.get(text[j], 0)
            if depth == 0:
                if text[match.end():j].strip():
                    answers.append(text[match.end():j].strip())
                break
    return answers


def extract_answer(text, thinks):
    """Last box after </think> for thinking models, first box otherwise. None if there is no answer."""
    if thinks:
        if "</think>" not in text:
            return None                       # still thinking when the token limit was reached
        boxes = boxed_answers(text.split("</think>")[-1])
        return boxes[-1] if boxes else None
    boxes = boxed_answers(text)
    return boxes[0] if boxes else None


def to_number(answer):
    """'$1,234' -> 1234.0, '20\\%' -> 20.0, '\\frac{3}{4}' -> 0.75, None if no number."""
    if answer is None:
        return None
    a = re.sub(r"\\[dt]?frac\{(\d+)\}\{(\d+)\}", r"\1/\2", answer).replace(",", "")
    if m := re.fullmatch(r"\s*(-?\d+)/(\d+)\s*", a):
        return int(m.group(1)) / int(m.group(2))
    numbers = re.findall(r"-?\d+(?:\.\d+)?", a)
    return float(numbers[-1]) if numbers else None


def is_correct(answer, gold):
    """True if the extracted answer is numerically equal to the gold answer."""
    value = to_number(answer)
    return value is not None and math.isclose(value, float(gold), abs_tol=1e-6)


def answer_cell(r, thinks):
    """Short text for the comparison table: '18 ✓', '180 ✗', 'cut off' or 'no answer'."""
    answer = extract_answer(r["output"], thinks)
    if answer is None:
        return "no answer" if r["finished"] else "cut off"
    return f"{answer}  {'✓' if is_correct(answer, r['gold']) else '✗'}"


def load_results(path):
    """Read a .jsonl results file; returns [] if it does not exist yet."""
    path = Path(path)
    if not path.exists():
        return []
    results = []
    for line in path.read_text(encoding="utf-8").splitlines():
        try:
            results.append(json.loads(line))
        except json.JSONDecodeError:
            pass    # a half-written line from a disconnect is ignored (that problem is simply redone)
    return results


def save_results(path, results):
    """Append results to a .jsonl file and force them to disk."""
    with open(path, "a", encoding="utf-8") as f:
        for r in results:
            f.write(json.dumps(r) + "\n")
        f.flush()
        os.fsync(f.fileno())    # make sure it really reaches Google Drive


def load_model(model_name):
    """Load model (float16, on GPU) and tokenizer set up for batched generation."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.padding_side = "left"           # needed for batched generation
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float16).to("cuda").eval()
    print(f"Loaded {model_name} | GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
    return model, tokenizer


def free_gpu():
    """Release GPU memory of objects that are no longer referenced."""
    gc.collect()
    torch.cuda.empty_cache()


class StopAfterAnswer(StoppingCriteria):
    """Stops each sequence as soon as it contains a complete, non-empty \\boxed{...} answer."""

    def __init__(self, tokenizer, prompt_length):
        self.tokenizer, self.prompt_length = tokenizer, prompt_length

    def __call__(self, input_ids, scores, **kwargs):
        texts = self.tokenizer.batch_decode(input_ids[:, self.prompt_length:], skip_special_tokens=True)
        return torch.tensor([bool(boxed_answers(t)) for t in texts], device=input_ids.device)


class StepTimer(StoppingCriteria):
    """Never stops; records the time after every generation step (times[k] = when token k was generated)."""

    def __init__(self):
        self.times = []

    def __call__(self, input_ids, scores, **kwargs):
        self.times.append(time.time())
        return torch.zeros(input_ids.shape[0], dtype=torch.bool, device=input_ids.device)


@torch.no_grad()
def generate_answers(model, tokenizer, questions, cfg, instruction, temperature, top_p):
    """Generate one answer per question. Returns a list of (text, n_tokens, finished, answer_time)."""
    prompts = [build_prompt(tokenizer, q, cfg["chat_template"], instruction) for q in questions]
    inputs = tokenizer(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
    prompt_length = inputs["input_ids"].shape[1]

    # Stop at the normal end token, and also at <|im_end|> (end of a chat turn for Qwen models)
    stop_ids = [tokenizer.eos_token_id]
    im_end = tokenizer.convert_tokens_to_ids("<|im_end|>")
    if isinstance(im_end, int) and im_end != tokenizer.unk_token_id:
        stop_ids.append(im_end)

    if cfg["sampling"]:
        decoding = dict(do_sample=True, temperature=temperature, top_p=top_p)
    else:
        decoding = dict(do_sample=False, temperature=None, top_p=None, top_k=None)   # greedy
    timer = StepTimer()
    criteria = [timer] + ([StopAfterAnswer(tokenizer, prompt_length)] if cfg["stop_after_answer"] else [])
    t0 = time.time()
    outputs = model.generate(
        **inputs, **decoding,
        max_new_tokens=cfg["max_new_tokens"],
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=stop_ids,
        stopping_criteria=StoppingCriteriaList(criteria),
    )

    results = []
    for tokens in outputs[:, prompt_length:].tolist():
        # Everything after the first stop token is padding.
        end = next((k for k, t in enumerate(tokens) if t in stop_ids), None)
        n_tokens = end if end is not None else len(tokens)
        text = tokenizer.decode(tokens[:n_tokens], skip_special_tokens=True)
        stopped_after_answer = cfg["stop_after_answer"] and bool(boxed_answers(text))
        finished = end is not None or stopped_after_answer or n_tokens < cfg["max_new_tokens"]
        # Time until this sequence produced its last token (its stop token, or the last token if it had none).
        answer_time = timer.times[end if end is not None else n_tokens - 1] - t0
        results.append((text, n_tokens, finished, answer_time))
    return results


def run_model(model_key, cfg, problems, results_dir, instruction, batch_size, seed, temperature, top_p):
    """Run one model on all problems not yet in `results_dir/<model_key>.jsonl`, saving after every batch."""
    path = Path(results_dir) / f"{model_key}.jsonl"
    done_ids = {r["id"] for r in load_results(path)}
    todo = [p for p in problems if p["id"] not in done_ids]
    print(f"{model_key}: {len(done_ids)} problems already done, {len(todo)} to go. Saving to {path}")
    if not todo:
        return

    model, tokenizer = load_model(cfg["name"])
    set_seed(seed)
    for start in range(0, len(todo), batch_size):
        batch = todo[start:start + batch_size]
        t0 = time.time()
        outputs = generate_answers(model, tokenizer, [p["question"] for p in batch], cfg, instruction, temperature, top_p)
        batch_time = time.time() - t0         # one generate call for the whole batch, so this is per batch
        results = []
        for p, (text, n_tokens, finished, answer_time) in zip(batch, outputs):
            answer = extract_answer(text, cfg["thinks"])
            results.append({**p, "model": cfg["name"], "output": text, "answer": answer,
                            "correct": is_correct(answer, p["gold"]),
                            "n_tokens": n_tokens, "finished": finished, "batch_time": batch_time,
                            "answer_time": answer_time, "chain_of_thought_length": text.count("\n\n")})
        save_results(path, results)
        print(f"  saved {len(done_ids) + start + len(batch)}/{len(problems)}  ({batch_time:.0f} s)")
    del model, tokenizer
    free_gpu()

    # Short sanity check, so problems with the setup are noticed right away.
    results = load_results(path)
    print(f"Done: {sum(r['correct'] for r in results)}/{len(results)} correct | "
          f"{sum(not r['finished'] for r in results)} cut off at the token limit | "
          f"{sum(r['answer'] is None for r in results)} without a boxed answer")


# Quick sanity check of the answer parsing
assert extract_answer("thinking... </think> so \\boxed{18}.", thinks=True) == "18"
assert extract_answer("thinking \\boxed{5} but ran out of tokens", thinks=True) is None
assert extract_answer("\\boxed{7}. Please put your answer within \\boxed{}. \\boxed{3}", thinks=False) == "7"
assert extract_answer("no box here", thinks=False) is None
assert is_correct("\\$1,234", "1234") and is_correct("20\\%", "20") and is_correct("75.00", "75")
assert to_number("\\frac{3}{4}") == 0.75 and not is_correct(None, "5")
print("Answer parsing works.")

Answer parsing works.


## 4. Run model 1: DeepSeek-R1-Distill-Qwen-1.5B
The reasoning model writes long "thinking" before answering. A batch takes as long as its longest answer, so expect roughly 5 minutes per batch of 16 (about 20 minutes for 50 problems).

In [7]:
run_model("r1-distill", MODELS["r1-distill"], problems, RESULTS_DIR, INSTRUCTION, BATCH_SIZE, SEED, TEMPERATURE, TOP_P)

r1-distill: 0 problems already done, 50 to go. Saving to /content/drive/MyDrive/VIS4AI/results/run_4_2026-11-08_15-40/r1-distill.jsonl


[transformers] `tokenizer_class` is `LlamaTokenizerFast` but `tokenizer.json` is ByteLevel; loading `TokenizersBackend`. Set `tokenizer_class` to `TokenizersBackend` in `tokenizer_config.json`.


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loaded deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B | GPU memory used: 3.6 GB
  saved 12/50  (1394 s)
  saved 24/50  (1387 s)
  saved 36/50  (1009 s)
  saved 48/50  (681 s)
  saved 50/50  (50 s)
Done: 43/50 correct | 3 cut off at the token limit | 3 without a boxed answer


## 5. Run model 2: Qwen2.5-Math-1.5B
Same problems. The first model was removed from the GPU, so there is room for this one. This model is much faster (short answers, stopped right after its `\boxed{}` answer).

In [8]:
run_model("qwen-math", MODELS["qwen-math"], problems, RESULTS_DIR, INSTRUCTION, BATCH_SIZE, SEED, TEMPERATURE, TOP_P)

qwen-math: 0 problems already done, 50 to go. Saving to /content/drive/MyDrive/VIS4AI/results/run_4_2026-11-08_15-40/qwen-math.jsonl


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loaded Qwen/Qwen2.5-Math-1.5B | GPU memory used: 3.1 GB
  saved 12/50  (32 s)
  saved 24/50  (130 s)
  saved 36/50  (20 s)
  saved 48/50  (131 s)
  saved 50/50  (15 s)
Done: 40/50 correct | 2 cut off at the token limit | 2 without a boxed answer


## 6. Look at the answers
Both models side by side, one row per problem (read back from Drive, so this also works after a disconnect).

`18 ✓` = correct, `180 ✗` = wrong, `cut off` = hit the token limit before answering, `no answer` = finished without a `\boxed{}` answer.
The answers are re-extracted from the saved outputs here, so improving `extract_answer` never requires re-running the models.

In [9]:
comparison = pd.DataFrame(problems).set_index("id")[["gold"]]
for key, cfg in MODELS.items():
    results = {r["id"]: r for r in load_results(RESULTS_DIR / f"{key}.jsonl")}
    comparison[key] = [answer_cell(results[i], cfg["thinks"]) if i in results else "not run"
                       for i in comparison.index]

pd.set_option("display.max_rows", None)
comparison

,gold,r1-distill,qwen-math
id,,,
0,18,18 ✓,18 ✓
1,3,3 ✓,3 ✓
2,70000,70000 ✓,70000 ✓
3,540,180 ✗,180 ✗
4,20,cut off,140 ✗
5,64,64 ✓,64 ✓
6,260,260 ✓,260 ✓
7,160,140 ✗,140 ✗
8,45,cut off,315 ✗


In [10]:
# Read one full output. Change these two values to look at other problems / the other model.
MODEL_KEY  = "r1-distill"    # "r1-distill" or "qwen-math"
PROBLEM_ID = 0

example = {r["id"]: r for r in load_results(RESULTS_DIR / f"{MODEL_KEY}.jsonl")}[PROBLEM_ID]
print("QUESTION:", example["question"])
print(f"\nCORRECT ANSWER: {example['gold']} | "
      f"MODEL ANSWER: {extract_answer(example['output'], MODELS[MODEL_KEY]['thinks'])} | "
      f"{example['n_tokens']} tokens | finished: {example['finished']}")
print("\nMODEL OUTPUT:\n", example["output"])

QUESTION: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

CORRECT ANSWER: 18 | MODEL ANSWER: 18 | 955 tokens | finished: True

MODEL OUTPUT:
 Okay, so I have this problem here about Janet and her ducks. Let me try to figure it out step by step. 

First, let me read the problem again and make sure I understand all the details. Janet's ducks lay 16 eggs per day. She eats three eggs for breakfast every morning and uses four eggs to bake muffins for her friends. Then, she sells the remaining eggs at the farmers' market, which sells each egg for $2. The question is asking how much money she makes every day at the farmers' market.

Alright, so let me break it down. There are a few key pieces of information:

1. Ducks lay 16 eggs per day.
2. She eats 3 eggs for b